# Notebook 08: Hybrid Allergen Detection

## Overview

Combines rule-based and ML predictions for robust allergen detection.

## Workflow

1. Load pre-trained model using utility modules
2. Load hybrid configuration
3. Define rule-based patterns (import from text_processing)
4. Hybrid prediction logic
5. Evaluate on test set using utility evaluation functions
6. Save configuration

## 0. Setup & Imports

Using modular utility functions to reduce code duplication

In [1]:
import json
import torch
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

from transformers import AutoTokenizer, AutoModelForSequenceClassification
from sklearn.metrics import classification_report, f1_score

# Import project utility modules
from utils.data_utils import (
    load_labeled_data,
    create_stratified_splits,
    get_data_directories
)
from utils.text_processing import (
    rule_match,
    get_allergen_list,
    clean_html,
    allergens_to_binary,
    detect_allergens_rule_based,
    has_explicit_allergen_statement,
    parse_official_tags
)
from utils.model_utils import (
    load_model_and_tokenizer,
    load_hybrid_config,
    predict_ml,
    hybrid_predict,
    find_best_thresholds
)
from utils.evaluation import (
    print_classification_report,
    print_per_class_metrics,
    error_analysis,
    evaluate_rule_vs_official,
)

# Set seeds for reproducibility
import random
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

print("Imports completed successfully using utility modules!")

Imports completed successfully using utility modules!


## Setup Complete

All imports consolidated in the cell above, reducing code duplication across the notebook.

In [2]:
import os

# Use project directory structure for consistent paths
dirs = get_data_directories()
DATA_PATH = os.path.join(dirs["final"], "labeled_dataset_enhanced.csv")
BEST_THRESHOLDS_PATH = os.path.join(dirs["models"], "best_thresholds.npy")
HYBRID_CONFIG_PATH = os.path.join(dirs["models"], "hybrid_config.json")
MODEL_PATH = os.path.join(dirs["models"], "mobilebert_allergen_final/")

# Batch size for inference — keep small to fit within GPU memory
BATCH_SIZE = 8

# Load labeled data using utility function
df = load_labeled_data(DATA_PATH)

# Create clean_text from ingredients_text_en (same as notebook 07)
df["ingredients_cleaned"] = df["ingredients_text_en"].apply(clean_html)

# Parse labels using utility function
df["labels"] = df["detected_allergens"].apply(allergens_to_binary)

print(f"Dataset shape: {df.shape}")
print(f"Label distribution (per class): {np.array(df['labels'].tolist()).sum(axis=0)}")

# Free any cached GPU memory before starting inference
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    print(f"GPU memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GiB total, "
          f"{torch.cuda.memory_allocated(0) / 1e9:.2f} GiB allocated")

Dataset shape: (1057, 13)
Label distribution (per class): [495 102  89  15 277 358  77  33]
GPU memory: 8.22 GiB total, 0.00 GiB allocated


In [3]:
# Generate stratified train/val/test splits (same approach as notebook 04)
# using the utility function from utils.data_utils

X = df["ingredients_cleaned"].values
y = np.array(df["labels"].tolist())

train_texts, val_texts, test_texts, train_labels, val_labels, test_labels = create_stratified_splits(
    X, y,
    train_size=0.7,
    val_size=0.15,
    test_size=0.15,
    random_state=SEED
)

print(f"Train size: {len(train_texts)} (positive samples: {(np.array(train_labels).sum(axis=1)>0).sum()})")
print(f"Val size:   {len(val_texts)}   (positive samples: {(np.array(val_labels).sum(axis=1)>0).sum()})")
print(f"Test size:  {len(test_texts)}  (positive samples: {(np.array(test_labels).sum(axis=1)>0).sum()})")

Train size: 739 (positive samples: 496)
Val size:   159   (positive samples: 112)
Test size:  159  (positive samples: 103)


## 3. Rule‑Based System

Using the robust rule_match function from text_processing utility

In [4]:
# The rule_match function is imported from utils.text_processing
# It includes comprehensive allergen keywords and negation handling
# No need to redefine it here - using the utility version ensures consistency

# Example usage:
# rule_present = rule_match(text, "milk")
print("Using rule_match from utils.text_processing")
allergen_list = get_allergen_list()
print(f"Available allergens: {allergen_list}")

Using rule_match from utils.text_processing
Available allergens: ['milk', 'eggs', 'peanuts', 'tree_nuts', 'soy', 'wheat', 'fish', 'shellfish']


## 4. Load the Trained MobileBERT Model and Tokenizer

Using utility functions for model loading

In [5]:
# Load the trained MobileBERT model and tokenizer using utilities
# Note: The model must be trained first by running notebook 07

print(f"Loading model from {MODEL_PATH}...")
model, tokenizer, device = load_model_and_tokenizer(MODEL_PATH)

# Compute max_length from the training data lengths for consistency with notebook 07
token_lengths = [len(tokenizer.encode(t, truncation=True)) for t in df["ingredients_cleaned"]]
MAX_LENGTH = max(token_lengths)
print(f"Computed max sequence length: {MAX_LENGTH}")

# Load hybrid configuration (generated by notebook 07's training)
# This is the SINGLE source of truth for thresholds
try:
    hybrid_config = load_hybrid_config(HYBRID_CONFIG_PATH)
    ml_thresholds = np.array(hybrid_config.get("ml_thresholds", None))
    rule_conf_threshold = hybrid_config.get("rule_conf_threshold", 0.5)
    mode = hybrid_config.get("mode", "hard_override")
    per_allergen_modes = hybrid_config.get("per_allergen_modes", None)
    print(f"Loaded ML thresholds: {ml_thresholds}")
    print(f"Rule conf threshold: {rule_conf_threshold}")
    print(f"Mode: {mode}")
    if per_allergen_modes:
        print(f"Per-allergen modes: {per_allergen_modes}")
except FileNotFoundError:
    print(f"Hybrid config not found at {HYBRID_CONFIG_PATH}")
    print("Using default parameters — will compute thresholds from validation set")
    ml_thresholds = None
    rule_conf_threshold = 0.5
    mode = "hard_override"
    per_allergen_modes = None

Loading model from /home/Woof/Dev/ENHANCING-FOOD-LABEL-TRANSPARENCY-FOR-FILIPINO-CONSUMERS-THROUGH-AI-BASED-INGREDIENT-INTERPRETATION/models/mobilebert_allergen_final/...


Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.


Computed max sequence length: 540
Loaded ML thresholds: [0.06 0.01 0.11 0.01 0.11 0.04 0.12 0.08]
Rule conf threshold: 0.5
Mode: hard_override


## 5. Text Preprocessing Function

Simple preprocessing (can be enhanced with utilities if needed)

In [6]:
# Text preprocessing is already done in cell 6 (clean_html) and cell 11 (tokenizer)
# The model handles tokenization internally via predict_ml and hybrid_predict
print(f"Using {BATCH_SIZE} samples per batch for inference")
print(f"Using max sequence length: {MAX_LENGTH}")

Using 8 samples per batch for inference
Using max sequence length: 540


## 6. Load Optimal ML Thresholds

Using utility function to find optimal thresholds if not already saved

In [7]:
# Use thresholds from hybrid_config.json (single source of truth, generated by notebook 07)
if ml_thresholds is not None:
    best_thresholds = ml_thresholds
    print(f"Loaded per-class thresholds from hybrid_config.json: {best_thresholds}")
else:
    print("No thresholds in config — computing from validation set...")
    val_ml_preds, val_probs = predict_ml(
        val_texts, model, tokenizer, device,
        max_length=MAX_LENGTH, batch_size=BATCH_SIZE
    )
    best_thresholds = find_best_thresholds(val_probs, np.array(val_labels))
    print("Computed thresholds:", best_thresholds)

Loaded per-class thresholds from hybrid_config.json: [0.06 0.01 0.11 0.01 0.11 0.04 0.12 0.08]


## 7. Evaluate on Test Set

Using utility evaluation functions for consistent reporting

In [8]:
# ML only baseline — uses batched inference to avoid GPU OOM
print(f"Running ML inference with batch_size={BATCH_SIZE}, max_length={MAX_LENGTH}...")
ml_test_preds, ml_test_probs = predict_ml(
    test_texts, model, tokenizer, device,
    thresholds=best_thresholds,
    max_length=MAX_LENGTH,
    batch_size=BATCH_SIZE
)
print("ML Only evaluation:")
print_classification_report(test_labels, ml_test_preds, get_allergen_list(), prefix="ML Only")

# Hybrid (hard override with global optimal threshold)
print("\nRunning hybrid (hard override) inference...")
test_hybrid_preds, _ = hybrid_predict(
    test_texts, model, tokenizer, device,
    thresholds=best_thresholds,
    rule_conf_threshold=rule_conf_threshold,
    mode='hard_override',
    max_length=MAX_LENGTH,
    batch_size=BATCH_SIZE,
    per_allergen_modes=per_allergen_modes
)
print("Hybrid (hard override) evaluation:")
print_classification_report(test_labels, test_hybrid_preds, get_allergen_list(), prefix="Hybrid (hard override)")

# Try alternative mode: high confidence bypass
print("\nRunning hybrid (high confidence bypass) inference...")
test_hybrid_bypass, _ = hybrid_predict(
    test_texts, model, tokenizer, device,
    thresholds=best_thresholds,
    rule_conf_threshold=rule_conf_threshold,
    mode='high_confidence_bypass',
    max_length=MAX_LENGTH,
    batch_size=BATCH_SIZE,
    per_allergen_modes=per_allergen_modes
)
print("Hybrid (high confidence bypass) evaluation:")
print_classification_report(test_labels, test_hybrid_bypass, get_allergen_list(), prefix="Hybrid (high confidence bypass)")

Running ML inference with batch_size=8, max_length=540...
ML Only evaluation:
=== ML Only ===
              precision    recall  f1-score   support

        milk     0.9861    0.9467    0.9660        75
        eggs     0.9333    0.8750    0.9032        16
     peanuts     1.0000    1.0000    1.0000        14
   tree_nuts     0.5000    0.3333    0.4000         3
         soy     0.8750    1.0000    0.9333        42
       wheat     0.9608    0.9074    0.9333        54
        fish     0.9167    0.9167    0.9167        12
   shellfish     1.0000    0.6000    0.7500         5

   micro avg     0.9447    0.9276    0.9361       221
   macro avg     0.8965    0.8224    0.8503       221
weighted avg     0.9458    0.9276    0.9342       221
 samples avg     0.5941    0.5968    0.5899       221

Macro F1: 0.8503
Micro F1: 0.9361


Running hybrid (hard override) inference...
Hybrid (hard override) evaluation:
=== Hybrid (hard override) ===
              precision    recall  f1-score   support



## 8. Error Analysis (examples where hybrid differs from ML)

Using utility function for error analysis

In [9]:
results_df = pd.DataFrame({
    "text": test_texts,
    "true": [list(l) for l in test_labels],
    "ml_pred": [list(p) for p in ml_test_preds],
    "hybrid_pred": [list(p) for p in test_hybrid_preds]
})
differences = results_df[results_df["ml_pred"] != results_df["hybrid_pred"]]
print(f"Number of test samples where hybrid changed prediction: {len(differences)}")
if len(differences) > 0:
    print("\nFirst 3 examples:")
    allergen_list = get_allergen_list()
    for i in range(min(3, len(differences))):
        row = differences.iloc[i]
        true_all = [allergen_list[j] for j, v in enumerate(row['true']) if v == 1]
        ml_all = [allergen_list[j] for j, v in enumerate(row['ml_pred']) if v == 1]
        hybrid_all = [allergen_list[j] for j, v in enumerate(row['hybrid_pred']) if v == 1]
        print(f"\nText: {row['text'][:150]}...")
        print(f"True: {true_all}")
        print(f"ML:   {ml_all}")
        print(f"Hybrid:{hybrid_all}")

Number of test samples where hybrid changed prediction: 0


## 9. Save hybrid configuration for production

Saving the current hybrid configuration

In [10]:
hybrid_config = {
    "ml_thresholds": best_thresholds.tolist(),
    "rule_conf_threshold": rule_conf_threshold,
    "mode": mode,
    "per_allergen_modes": per_allergen_modes or {"tree_nuts": "rule_priority"}
}
config_path = os.path.join(dirs["models"], "hybrid_config.json")
with open(config_path, "w") as f:
    json.dump(hybrid_config, f, indent=2)
print(f"Hybrid configuration saved to {config_path}")

Hybrid configuration saved to /home/Woof/Dev/ENHANCING-FOOD-LABEL-TRANSPARENCY-FOR-FILIPINO-CONSUMERS-THROUGH-AI-BASED-INGREDIENT-INTERPRETATION/models/hybrid_config.json


## 10. Official Tag Evaluation (Phase 1)

Compare rule-based allergen detection against OFF (Open Food Facts) official tags.
This measures the real-world precision of the keyword matching after Phase 4 refinements
(context-aware ambiguity handling, negation strengthening, exemption rules).

Baseline was 21.88% exact agreement on the 96 products with explicit allergen statements.

In [11]:
# ── Phase 1: Rule-based vs Official Tag Evaluation ──

from utils.text_processing import (
    detect_allergens_rule_based, has_explicit_allergen_statement, clean_html, parse_official_tags
)
from utils.evaluation import evaluate_rule_vs_official

# Load cleaned dataset (has raw ingredients_text_en AND allergens_tags)
# NOTE: CLEANED_PATH instead of DATA_PATH — the labeled dataset drops the original OFF tags
CLEANED_PATH = os.path.join(dirs["processed"], "cleaned_dataset.csv")
df_official = load_labeled_data(CLEANED_PATH)

print(f"Loaded {len(df_official)} products from cleaned dataset")
print(f"  - Non-empty allergens_tags: {df_official['allergens_tags'].dropna().astype(str).str.len().gt(2).sum()}")

# Run rule-based detection on the full ingredient text
allergen_list = get_allergen_list()
df_official["detected_allergens"] = df_official["ingredients_text_en"].apply(
    lambda x: detect_allergens_rule_based(str(x).lower())
)

# Parse official allergen tags (uses regex extraction to handle numpy-style format)
df_official["official_allergens_mapped"] = df_official["allergens_tags"].apply(
    lambda x: parse_official_tags(x) if pd.notna(x) and str(x) not in ('[]', 'nan', '') else []
)

# ─── Evaluation A: Full dataset (1057 samples) ───
print("=" * 60)
print("  EVALUATION A: Full dataset — all 1057 products")
print("=" * 60)
full_results = evaluate_rule_vs_official(df_official, allergen_list, prefix="Rule vs Official (All 1057)")

# ─── Evaluation B: 96+ products with explicit allergen statements ───
print("=" * 60)
print("  EVALUATION B: Products with explicit allergen statements")
print("  Baseline (from NB06 pre-Phase-4): exact agreement = 21.88%")
print("=" * 60)
df_official["explicit_allergen_statement"] = df_official["ingredients_text_en"].apply(
    has_explicit_allergen_statement
)
eval_df = df_official[df_official["explicit_allergen_statement"]]
print(f"\n  Filtered to {len(eval_df)} products with explicit statements.\n")

explicit_results = evaluate_rule_vs_official(eval_df, allergen_list, prefix="Rule vs Official (Explicit)")

# Improved from baseline?
baseline_exact = 0.2188
improvement = explicit_results["exact_agreement"] - baseline_exact
print(f"  Improvement from baseline: {improvement:+.2%} ({baseline_exact:.1%} → {explicit_results['exact_agreement']:.1%})")
if explicit_results["exact_agreement"] >= 0.50:
    print(f"  ✓ TARGET MET: exact agreement ≥ 50% (target was >50%)")
else:
    print(f"  ✗ Target not met: {explicit_results['exact_agreement']:.1%} < 50%")

# ─── Accuracy check: are official tags complete? ───
# Some official tags might list fewer allergens than the product actually contains.
# Measure how often detected ⊆ official (no false positives / rule is a subset).
# High superset agreement means rules rarely miss allergens listed officially.
print(f"\n  ├─ Exact match:     {explicit_results['exact_agreement']:.1%}")
print(f"  ├─ No false pos:    {explicit_results['subset_agreement']:.1%}")
print(f"  ├─ No false neg:    {explicit_results['superset_agreement']:.1%}")
print(f"  └─ Avg Jaccard:     {explicit_results['avg_jaccard']:.1%}")

Loaded 1057 products from cleaned dataset
  - Non-empty allergens_tags: 765
  EVALUATION A: Full dataset — all 1057 products

  Rule vs Official (All 1057)
  Samples evaluated: 1057
  Exact match (strict):        76.63%
  Detected ⊆ official (no FP): 86.47%
  Detected ⊇ official (no FN): 88.93%
  Avg Jaccard:                 87.35%

  Per-class metrics:
  Allergen      Precision     Recall         F1
  ────────────────────────────────────────────
  milk            89.40%    95.11%    92.16%
  eggs            83.85%    94.78%    88.98%
  peanuts         70.79%    95.45%    81.29%
  tree_nuts       70.77%    90.20%    79.31%
  soy             92.35%    89.32%    90.81%
  wheat           92.66%    92.41%    92.54%
  fish            85.71%    80.49%    83.02%
  shellfish       82.35%    63.64%    71.79%

  EVALUATION B: Products with explicit allergen statements
  Baseline (from NB06 pre-Phase-4): exact agreement = 21.88%

  Filtered to 96 products with explicit statements.


  Rule vs Off